# 06 · Rates: Swaps, FRAs, and Basis Trades

This notebook shows how the rates pricers cover deposits, forward rate agreements, vanilla swaps, and basis structures off the same market context.

### Learning Objectives
- Build a single USD SOFR market with discount and multiple forward curves
- Price deposits and FRAs with carry, par rate, and PV01 metrics
- Instantiate a receive-fixed swap and inspect annuity/DV01 outputs
- Value basis swaps linking two floating indices and inspect sensitivity

In [ ]:
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD
from finstack.core.dates.daycount import DayCount
from finstack.core.dates.tenor import Tenor
from finstack.core.market_data.context import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve, ForwardCurve
from finstack.valuations.instruments import (
    Deposit,
    ForwardRateAgreement,
    InterestRateSwap,
    BasisSwap,
    BasisSwapLeg,
)
from finstack.valuations.pricer import create_standard_registry

as_of = date(2024, 1, 2)

market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9975), (1.0, 0.9940), (3.0, 0.9700), (5.0, 0.9350)],
    )
)
market.insert_forward(
    ForwardCurve(
        "USD-SOFR-3M",
        0.25,
        [(0.0, 0.0350), (1.0, 0.0360), (3.0, 0.0385), (5.0, 0.0400)],
        base_date=as_of,
    )
)
market.insert_forward(
    ForwardCurve(
        "USD-SOFR-6M",
        0.5,
        [(0.0, 0.0355), (1.0, 0.0370), (3.0, 0.0390), (5.0, 0.0410)],
        base_date=as_of,
    )
)

registry = create_standard_registry()
notional = Money(1_000_000, USD)


## 1. Deposits & FRAs
Deposits capture short-dated funding trades; FRAs lock in future fixings. Both rely on the same discount/forward curves and expose par rate, PV, and PV01 metrics.

In [ ]:
deposit = (
    Deposit.builder("USD-DEP-3M")
    .money(Money(5_000_000, USD))
    .start(as_of)
    .maturity(as_of + timedelta(days=92))
    .day_count(DayCount.ACT_360)
    .disc_id("USD-OIS")
    .quote_rate(0.0450)
    .build()
)
res_dep = registry.price(deposit, "discounting", market, as_of=as_of)
print("Deposit PV:", res_dep.value)

fra = (
    ForwardRateAgreement.builder("USD-FRA-3x6")
    .money(notional)
    .fixed_rate(0.0360)
    .fixing_date(as_of + timedelta(days=30))
    .start_date(as_of + timedelta(days=92))
    .end_date(as_of + timedelta(days=182))
    .disc_id("USD-OIS")
    .fwd_id("USD-SOFR-3M")
    .pay_fixed(False)
    .build()
)
fra_res = registry.price_with_metrics(fra, "discounting", market, ["par_rate", "pv01"], as_of=as_of)
print("FRA PV:", fra_res.value)
print("FRA par rate:", fra_res.measures.get("par_rate"))
print("FRA PV01:", fra_res.measures.get("pv01"))


## 2. Plain-Vanilla Interest Rate Swap
Helper constructors create standard receive-fixed/pay-float SOFR swaps. Request annuity, DV01, and par rate metrics to support hedging and curve construction.

In [ ]:
swap = (
    InterestRateSwap.builder("USD-SOFR-SWAP")
    .money(notional)
    .side("receive_fixed")
    .fixed_rate(0.0325)
    .start(as_of + timedelta(days=2))
    .maturity(date(2029, 1, 2))
    .disc_id("USD-OIS")
    .fwd_id("USD-SOFR-3M")
    .build()
)
swap_res = registry.price_with_metrics(
    swap,
    "discounting",
    market,
    ["annuity", "dv01", "par_rate"],
    as_of=as_of,
)
print("Swap PV:", swap_res.value)
print("Annuity:", swap_res.measures.get("annuity"))
print("DV01:", swap_res.measures.get("dv01"))
print("Par rate:", swap_res.measures.get("par_rate"))


## 3. Basis Swap (SOFR 3M vs 6M)
Basis swaps exchange two floating legs referencing different tenors. Spreads are set per leg; DV01 helps explain relative-value positioning.

In [ ]:
start = as_of + timedelta(days=2)
maturity = date(2030, 1, 2)
primary_leg = BasisSwapLeg(
    "USD-SOFR-3M",
    discount_curve="USD-OIS",
    start_date=start,
    end_date=maturity,
    frequency="3m",
    calendar_id="usny",
    stub="short_front",
    spread_bp=0.0,
)
reference_leg = BasisSwapLeg(
    "USD-SOFR-6M",
    discount_curve="USD-OIS",
    start_date=start,
    end_date=maturity,
    frequency="6m",
    calendar_id="usny",
    stub="short_front",
    spread_bp=5.0,
)
basis = (
    BasisSwap.builder("USD-BASIS")
    .money(notional)
    .primary_leg(primary_leg)
    .reference_leg(reference_leg)
    .build()
)
basis_res = registry.price_with_metrics(basis, "discounting", market, ["dv01"], as_of=as_of)
print("Basis swap PV:", basis_res.value)
print("Basis swap DV01:", basis_res.measures.get("dv01"))


## Summary
- Funding instruments (deposits, FRAs) and swaps all pull from a shared `MarketContext`.
- Helper constructors simplify building USD SOFR swaps for hedging and curve calibration.
- Basis trades between 3M and 6M projections are supported through `BasisSwapLeg` objects with spreads and stub control.
- Metrics such as annuity, DV01, PV01, and par rate fall out directly from `price_with_metrics`.